# Fusion21 Full Data Pipeline / Fusion21 完整数据管道

**Primary executable delivery / 主要可执行交付文件**

This notebook runs the same validated processing logic used by the Streamlit
application. Run the cells from top to bottom, or select **Run All**.

本 Notebook 使用网站同一套经过测试的计算逻辑。按顺序运行全部单元格，
可以从原始数据一直生成网站读取的最终文件。

## TL;DR / 核心结果

- Public data are converted from **LSOA → LAD19 → nine English Regions**.
- IMD is population weighted at both aggregation stages.
- `Raw Social Need Index = (population-weighted IMD Score + unemployment rate) / 2`.
- `Activity Score = Yes responses / all possible responses × 100`.
- `Composite Contribution Score = (Activity Score + Foundation Score) / 2`.
- Contract value is displayed as procurement footprint and is not included in
  the social-contribution score.
- Fusion21 records in this prototype are synthetic and must not be interpreted
  as actual company performance.

## 1. Goal and workflow / 目标与流程

```text
Extract raw data
        ↓
Validate fields and missing values
        ↓
LSOA → LAD19 population weighting
        ↓
LAD19 → RGN19 population weighting
        ↓
Social need and synthetic contribution calculations
        ↓
Validate formulas and write app-ready CSV files
```

The notebook is the reader-facing entry point. `pipeline.py` remains the
reusable implementation used by both this notebook and the website, preventing
two versions of the formula from drifting apart.

## 2. Setup / 运行环境

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
import plotly.express as px
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

start_directory = Path.cwd().resolve()
project_candidates = [
    start_directory,
    start_directory / "04_地图分析软件工具" / "Fusion21_App",
]
project_directory = next(
    (
        candidate
        for candidate in project_candidates
        if (candidate / "pipeline.py").exists()
        and (candidate / "data").exists()
    ),
    None,
)
if project_directory is None:
    raise FileNotFoundError(
        "Could not find pipeline.py and data/. Open this notebook from the "
        "Fusion21 delivery folder or Fusion21_App folder."
    )

os.chdir(project_directory)
if str(project_directory) not in sys.path:
    sys.path.insert(0, str(project_directory))

import pipeline

FORCE_DOWNLOAD = False
PREVIEW_ROWS = 5
print(f"Project directory: {project_directory}")
print(f"Python executable: {sys.executable}")

Project directory: C:\Users\12622\Desktop\all\fusion21\01_网站程序_运行代码
Python executable: C:\Users\12622\Desktop\all\fusion21\.venv\Scripts\python.exe


In [2]:
source_catalogue = pd.DataFrame(
    [
        ["IMD 2019 File 7", "LSOA", "IMD Score, Rank, Decile and population"],
        ["ONS geography lookup", "LAD19 → RGN19", "Assign each LAD to an English Region"],
        ["ONS/Nomis labour market", "Region", "Latest available unemployment rate"],
        ["Synthetic contracts", "Contract", "Procurement footprint"],
        ["Synthetic activities", "Contract responses", "Activity Score"],
        ["Synthetic Foundation payments", "Payment", "Foundation Score"],
    ],
    columns=["Source", "Original level", "Use in pipeline"],
)
display(source_catalogue)

,Source,Original level,Use in pipeline
0,IMD 2019 File 7,LSOA,"IMD Score, Rank, Decile and population"
1,ONS geography lookup,LAD19 → RGN19,Assign each LAD to an English Region
2,ONS/Nomis labour market,Region,Latest available unemployment rate
3,Synthetic contracts,Contract,Procurement footprint
4,Synthetic activities,Contract responses,Activity Score
5,Synthetic Foundation payments,Payment,Foundation Score


## 3. Extract / 读取原始数据

`extract()` reads or refreshes all official inputs, boundaries and synthetic
Fusion21 tables. Raw inputs remain under `data/raw/`; generated intermediate
tables are kept under `data/interim/`.

In [3]:
extracted = pipeline.extract(force=FORCE_DOWNLOAD)

extract_summary = []
for name, value in extracted.items():
    if isinstance(value, pd.DataFrame):
        extract_summary.append([name, len(value), len(value.columns)])
    elif isinstance(value, dict):
        extract_summary.append([name, len(value), None])

display(
    pd.DataFrame(
        extract_summary,
        columns=["Extracted object", "Rows or contained objects", "Columns"],
    )
)

,Extracted object,Rows or contained objects,Columns
0,boundaries,2,NaN
1,imd_lad_raw,317,12.000
2,imd_region_raw,9,13.000
3,unemployment_raw,12,11.000
4,fusion21_synthetic,6,NaN


In [4]:
lsoa_columns = [
    pipeline.LSOA_CODE,
    pipeline.LAD_CODE,
    pipeline.LAD_NAME,
    pipeline.IMD_SCORE,
    pipeline.IMD_RANK,
    pipeline.IMD_DECILE,
    pipeline.TOTAL_POPULATION,
]
raw_lsoa = pd.read_csv(
    pipeline.IMD_FILE_7_CSV,
    usecols=lsoa_columns,
    low_memory=False,
)
for numeric_column in [
    pipeline.IMD_SCORE,
    pipeline.IMD_RANK,
    pipeline.IMD_DECILE,
    pipeline.TOTAL_POPULATION,
]:
    raw_lsoa[numeric_column] = pd.to_numeric(
        raw_lsoa[numeric_column], errors="coerce"
    )

print(f"Raw LSOA rows: {len(raw_lsoa):,}")
print(f"Distinct LAD19 areas: {raw_lsoa[pipeline.LAD_CODE].nunique():,}")
display(raw_lsoa.head(PREVIEW_ROWS))

Raw LSOA rows: 32,844
Distinct LAD19 areas: 317


,LSOA code (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners)
0,E01000001,E09000001,City of London,6.208,29199,9,1296
1,E01000002,E09000001,City of London,5.143,30379,10,1156
2,E01000003,E09000001,City of London,19.402,14915,5,1350
3,E01000005,E09000001,City of London,28.652,8678,3,1121
4,E01000006,E09000002,Barking and Dagenham,19.837,14486,5,2040


In [5]:
input_checks = pd.DataFrame(
    {
        "Check": [
            "Missing LSOA codes",
            "Duplicate LSOA codes",
            "Missing LAD19 codes",
            "Missing IMD Scores",
            "Missing population values",
            "Non-positive population values",
        ],
        "Count": [
            raw_lsoa[pipeline.LSOA_CODE].isna().sum(),
            raw_lsoa[pipeline.LSOA_CODE].duplicated().sum(),
            raw_lsoa[pipeline.LAD_CODE].isna().sum(),
            raw_lsoa[pipeline.IMD_SCORE].isna().sum(),
            raw_lsoa[pipeline.TOTAL_POPULATION].isna().sum(),
            raw_lsoa[pipeline.TOTAL_POPULATION].le(0).sum(),
        ],
    }
)
display(input_checks)

,Check,Count
0,Missing LSOA codes,0
1,Duplicate LSOA codes,0
2,Missing LAD19 codes,0
3,Missing IMD Scores,0
4,Missing population values,0
5,Non-positive population values,0


## 4. First aggregation: LSOA → LAD19 / 第一次人口加权

A simple mean would give every LSOA the same influence. Population weighting
instead estimates the IMD Score experienced by the average resident:

$$
\text{LAD IMD Score}=
\frac{\sum(\text{LSOA IMD Score}\times\text{LSOA population})}
{\sum\text{LSOA population}}
$$

The next cell uses three real Manchester LSOAs to make the arithmetic visible.
The formal LAD result then uses every available Manchester LSOA.

In [6]:
manchester_sample = (
    raw_lsoa.loc[raw_lsoa[pipeline.LAD_NAME].eq("Manchester")]
    .head(3)
    .copy()
)
if len(manchester_sample) != 3:
    raise ValueError("Expected at least three Manchester LSOAs in IMD File 7.")

manchester_sample["IMD Score × Population"] = (
    manchester_sample[pipeline.IMD_SCORE]
    * manchester_sample[pipeline.TOTAL_POPULATION]
)
sample_weighted_score = (
    manchester_sample["IMD Score × Population"].sum()
    / manchester_sample[pipeline.TOTAL_POPULATION].sum()
)

display(
    manchester_sample[
        [
            pipeline.LSOA_CODE,
            pipeline.LAD_NAME,
            pipeline.IMD_SCORE,
            pipeline.TOTAL_POPULATION,
            "IMD Score × Population",
        ]
    ]
)
print(
    "Three-row demonstration: "
    f"{manchester_sample['IMD Score × Population'].sum():,.3f} ÷ "
    f"{manchester_sample[pipeline.TOTAL_POPULATION].sum():,.0f} "
    f"= {sample_weighted_score:,.3f}"
)

,LSOA code (2011),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Total population: mid 2015 (excluding prisoners),IMD Score × Population
4935,E01005061,Manchester,62.256,1560,"97,119.360"
4936,E01005062,Manchester,23.458,3143,"73,728.494"
4937,E01005063,Manchester,58.123,1743,"101,308.389"


Three-row demonstration: 272,156.243 ÷ 6,446 = 42.221


In [7]:
manchester_all = raw_lsoa.loc[
    raw_lsoa[pipeline.LAD_NAME].eq("Manchester")
].copy()
manchester_all["weighted_score"] = (
    manchester_all[pipeline.IMD_SCORE]
    * manchester_all[pipeline.TOTAL_POPULATION]
)
recalculated_manchester_score = (
    manchester_all["weighted_score"].sum()
    / manchester_all[pipeline.TOTAL_POPULATION].sum()
)
pipeline_manchester = extracted["imd_lad_raw"].loc[
    extracted["imd_lad_raw"]["LAD19NM"].eq("Manchester"),
    ["LAD19CD", "LAD19NM", "lsoa_count", "population_total", "mean_imd_score"],
]

display(pipeline_manchester)
print(f"Recalculated from all Manchester LSOAs: {recalculated_manchester_score:,.3f}")

,LAD19CD,LAD19NM,lsoa_count,population_total,mean_imd_score
159,E08000003,Manchester,282,529282,40.005


Recalculated from all Manchester LSOAs: 40.005


## 5. Second aggregation: LAD19 → RGN19 / 第二次人口加权

The IMD source identifies LAD19 areas but does not contain the final nine-region
field. The official lookup assigns every LAD19 to one RGN19. The LAD results are
then weighted by each LAD's total population.

This is not counting population twice. It is a hierarchical weighted average,
and it gives the same regional result as aggregating all underlying LSOAs directly.

In [8]:
lookup_path = pipeline.RAW_DIR / "geography" / "lad19_to_rgn19_lookup.csv"
lad_region_lookup = pd.read_csv(lookup_path)
example_lads = ["Manchester", "Salford", "Liverpool", "Cheshire East"]
display(
    lad_region_lookup.loc[
        lad_region_lookup["LAD19NM"].isin(example_lads),
        ["LAD19CD", "LAD19NM", "RGN19CD", "RGN19NM"],
    ].sort_values("LAD19NM")
)

,LAD19CD,LAD19NM,RGN19CD,RGN19NM
138,E06000049,Cheshire East,E12000002,North West
148,E08000012,Liverpool,E12000002,North West
149,E08000003,Manchester,E12000002,North West
156,E08000006,Salford,E12000002,North West


In [9]:
lad_with_region = extracted["imd_lad_raw"].merge(
    lad_region_lookup,
    on=["LAD19CD", "LAD19NM"],
    how="left",
    validate="one_to_one",
)
north_west_lads = lad_with_region.loc[
    lad_with_region["RGN19NM"].eq("North West")
].copy()
north_west_lads["LAD score × LAD population"] = (
    north_west_lads["mean_imd_score"]
    * north_west_lads["population_total"]
)
north_west_score = (
    north_west_lads["LAD score × LAD population"].sum()
    / north_west_lads["population_total"].sum()
)
pipeline_north_west = extracted["imd_region_raw"].loc[
    extracted["imd_region_raw"]["RGN19NM"].eq("North West"),
    [
        "RGN19CD",
        "RGN19NM",
        "local_authority_count",
        "lsoa_count",
        "population_total",
        "mean_imd_score",
    ],
]

display(
    north_west_lads.loc[
        north_west_lads["LAD19NM"].isin(example_lads),
        [
            "LAD19CD",
            "LAD19NM",
            "mean_imd_score",
            "population_total",
            "LAD score × LAD population",
        ],
    ].sort_values("LAD19NM")
)
display(pipeline_north_west)
print(f"North West recalculated from all LADs: {north_west_score:,.3f}")

,LAD19CD,LAD19NM,mean_imd_score,population_total,LAD score × LAD population
53,E06000049,Cheshire East,14.475,375457,"5,434,802.953"
154,E08000012,Liverpool,42.412,479723,"20,345,877.690"
159,E08000003,Manchester,40.005,529282,"21,173,669.745"
219,E08000006,Salford,34.210,244530,"8,365,387.917"


,RGN19CD,RGN19NM,local_authority_count,lsoa_count,population_total,mean_imd_score
4,E12000002,North West,39,4497,7166543,28.090


North West recalculated from all LADs: 28.090


## 6. Transform public indicators / 计算社会需求

Neither component is Min-Max standardised. The values therefore remain connected
to the observed IMD Score and unemployment rate and are not forced to 0 and 100.

In [10]:
transformed = pipeline.transform(extracted)
need_table = transformed["composite_need"][
    ["area_code", "area_name", "mean_imd_score", "unemployment_rate", "value", "period"]
].rename(columns={"value": "raw_social_need_index"})
need_table = need_table.sort_values(
    "raw_social_need_index", ascending=False
).reset_index(drop=True)

formula_check = (
    (need_table["mean_imd_score"] + need_table["unemployment_rate"]) / 2
    - need_table["raw_social_need_index"]
).abs()
if formula_check.max() > 0.11:
    raise AssertionError("The social-need formula check failed.")

display(need_table)

,area_code,area_name,mean_imd_score,unemployment_rate,raw_social_need_index,period
0,E12000001,North East,28.000,5.400,16.700,IMD 2019 + Feb 2026-Apr 2026
1,E12000002,North West,28.100,5.000,16.600,IMD 2019 + Feb 2026-Apr 2026
2,E12000003,Yorkshire and The Humber,26.000,5.800,15.900,IMD 2019 + Feb 2026-Apr 2026
3,E12000005,West Midlands,25.300,5.300,15.300,IMD 2019 + Feb 2026-Apr 2026
4,E12000007,London,21.800,6.600,14.200,IMD 2019 + Feb 2026-Apr 2026
5,E12000004,East Midlands,20.400,5.500,13.000,IMD 2019 + Feb 2026-Apr 2026
6,E12000009,South West,18.200,4.400,11.300,IMD 2019 + Feb 2026-Apr 2026
7,E12000006,East of England,17.400,4.200,10.800,IMD 2019 + Feb 2026-Apr 2026
8,E12000008,South East,15.500,3.800,9.700,IMD 2019 + Feb 2026-Apr 2026


In [11]:
need_figure = px.bar(
    need_table.sort_values("raw_social_need_index"),
    x="raw_social_need_index",
    y="area_name",
    orientation="h",
    title="Raw Social Need Index by English Region",
    labels={
        "raw_social_need_index": "Raw Social Need Index",
        "area_name": "Region",
    },
    color="raw_social_need_index",
    color_continuous_scale="YlOrRd",
)
need_figure.update_layout(height=480, coloraxis_showscale=False)
need_figure.show()

## 7. Transform synthetic Fusion21 inputs / 计算模拟贡献

The three input tables represent contracts, social-value activity responses and
Foundation payments. They are synthetic test records, but their schemas demonstrate
how future non-sensitive Fusion21 files enter the same workflow.

In [12]:
synthetic = transformed["fusion21_synthetic"]

print("Synthetic contracts")
display(synthetic["contracts"].head(3))
print("Synthetic activity responses")
display(synthetic["activities"].head(3))
print("Synthetic Foundation payments")
display(synthetic["foundation"].head(3))

Synthetic contracts


,ID Number,Member,Supplier,Fusion21 Framework,Lot,Start Date,Contract Value FY2024/25,Delivery Postcode,Local Authority Code,Local Authority Name,ONS Region Code,Region Name,Latitude,Longitude,Synthetic Data
0,F21-SYN-0001,Demo Member 01-1,Demo Supplier 01,Decarbonisation,Lot 1,2024-03-04,"2,994,000.000",NE1 1AA,E08000021,Newcastle upon Tyne,E12000001,North East,55.006,-1.518,True
1,F21-SYN-0002,Demo Member 01-2,Demo Supplier 02,Construction Works,Lot 2,2024-05-07,"3,534,000.000",NE1 1AA,E08000021,Newcastle upon Tyne,E12000001,North East,55.039,-1.532,True
2,F21-SYN-0003,Demo Member 01-1,Demo Supplier 03,Facilities Management,Lot 3,2024-07-10,"1,804,000.000",NE1 1AA,E08000021,Newcastle upon Tyne,E12000001,North East,54.905,-1.591,True


Synthetic activity responses


,Fusion21 Contract ID,Fusion21 Member,Supplier Name,Synthetic Data,Taken steps to reduce carbon emissions or incorporate renewable energy into your operations? (e.g. switching to electric vehicles),Taken steps to reduce waste e.g. supporting community-level recycling,Implemented measures to reduce water consumption? (e.g. rainwater harvesting),"Protected local ecosystems? (e.g. by planting trees, creating green spaces)",Actively engaged with local communities/schools/colleges to promote the benefits of renewable energy?,Delivered training initiatives that upskills existing members of staff around sustainability/green skills?
0,F21-SYN-0001,Demo Member 01-1,Demo Supplier 01,True,No,Yes,No,No,No,Yes
1,F21-SYN-0002,Demo Member 01-2,Demo Supplier 02,True,No,Yes,No,No,Yes,No
2,F21-SYN-0003,Demo Member 01-1,Demo Supplier 03,True,No,No,No,No,No,No


Synthetic Foundation payments


,Fusion21 Contract ID,Fusion21 Member,Month Paid,Associated Foundation Budget,Programme Budget,Associated Foundation Case,Amount,ONS Region Code,Region Name,Synthetic Data
0,F21-SYN-0001,Demo Member 01-1,2024-06,Community Wellbeing,"880,000.000",F21F-SYN-0001,"93,000.000",E12000001,North East,True
1,F21-SYN-0002,Demo Member 01-2,2024-08,Environmental Sustainability,"609,000.000",F21F-SYN-0002,"67,000.000",E12000001,North East,True
2,F21-SYN-0003,Demo Member 01-1,2024-10,Financial Inclusion,"490,000.000",F21F-SYN-0003,"67,000.000",E12000001,North East,True


In [13]:
contribution_table = synthetic["region_summary"][
    [
        "area_code",
        "area_name",
        "project_count",
        "contract_value",
        "recorded_activity_count",
        "activity_possible_count",
        "activity_score",
        "foundation_investment",
        "foundation_score",
        "contribution_score",
    ]
].copy()

activity_check = (
    contribution_table["recorded_activity_count"]
    / contribution_table["activity_possible_count"]
    * 100
)
contribution_check = (
    contribution_table["activity_score"]
    + contribution_table["foundation_score"]
) / 2
if (activity_check - contribution_table["activity_score"]).abs().max() > 0.11:
    raise AssertionError("The Activity Score formula check failed.")
if (contribution_check - contribution_table["contribution_score"]).abs().max() > 0.11:
    raise AssertionError("The contribution formula check failed.")

display(contribution_table.sort_values("contribution_score", ascending=False))
print("Contract value is retained as procurement footprint only.")

,area_code,area_name,project_count,contract_value,recorded_activity_count,activity_possible_count,activity_score,foundation_investment,foundation_score,contribution_score
6,E12000007,London,7,"45,622,000.000",27,42,64.300,"648,000.000",100.000,82.100
4,E12000005,West Midlands,6,"18,855,000.000",23,36,63.900,"513,500.000",74.700,69.300
1,E12000002,North West,6,"23,061,000.000",24,36,66.700,"380,000.000",49.700,58.200
5,E12000006,East of England,5,"22,133,000.000",15,30,50.000,"440,500.000",61.000,55.500
7,E12000008,South East,7,"36,361,000.000",24,42,57.100,"294,500.000",33.600,45.400
8,E12000009,South West,4,"18,565,000.000",13,24,54.200,"261,000.000",27.300,40.700
3,E12000004,East Midlands,4,"16,032,000.000",11,24,45.800,"299,000.000",34.500,40.100
2,E12000003,Yorkshire and The Humber,3,"6,359,000.000",8,18,44.400,"115,500.000",0.000,22.200
0,E12000001,North East,3,"8,332,000.000",4,18,22.200,"227,000.000",20.900,21.600


Contract value is retained as procurement footprint only.


## 8. Compare need with contribution / 比较需求与贡献

The two composite measures use different units, so their raw values are not
subtracted. Instead, the nine regions are ranked separately and divided into
three equal groups: Low, Medium and High. Priority review means:

`High need tertile AND Low contribution tertile`.

In [14]:
def add_tertile(frame, value_column, output_column):
    result = frame.copy()
    ordered = result.sort_values(
        [value_column, "area_name"],
        ascending=[True, True],
        kind="mergesort",
    )
    positions = pd.Series(range(len(ordered)), index=ordered.index)
    ordered[output_column] = pd.qcut(
        positions, q=3, labels=["Low", "Medium", "High"]
    ).astype(str)
    result[output_column] = ordered[output_column]
    return result


comparison = need_table[
    ["area_code", "area_name", "raw_social_need_index"]
].merge(
    contribution_table[["area_code", "contribution_score"]],
    on="area_code",
    validate="one_to_one",
)
comparison = add_tertile(
    comparison, "raw_social_need_index", "need_tertile"
)
comparison = add_tertile(
    comparison, "contribution_score", "contribution_tertile"
)
comparison["priority_review"] = (
    comparison["need_tertile"].eq("High")
    & comparison["contribution_tertile"].eq("Low")
)

display(
    comparison.sort_values(
        ["priority_review", "raw_social_need_index"],
        ascending=[False, False],
    )
)
priority_regions = comparison.loc[
    comparison["priority_review"], "area_name"
].tolist()
print(f"Priority-review regions: {priority_regions or ['None']}")

,area_code,area_name,raw_social_need_index,contribution_score,need_tertile,contribution_tertile,priority_review
0,E12000001,North East,16.700,21.600,High,Low,True
2,E12000003,Yorkshire and The Humber,15.900,22.200,High,Low,True
1,E12000002,North West,16.600,58.200,High,High,False
3,E12000005,West Midlands,15.300,69.300,Medium,High,False
4,E12000007,London,14.200,82.100,Medium,High,False
5,E12000004,East Midlands,13.000,40.100,Medium,Low,False
6,E12000009,South West,11.300,40.700,Low,Medium,False
7,E12000006,East of England,10.800,55.500,Low,Medium,False
8,E12000008,South East,9.700,45.400,Low,Medium,False


Priority-review regions: ['North East', 'Yorkshire and The Humber']


## 9. Validate and load / 验证并输出

Validation checks region coverage, required columns and score formulas before any
app-ready output is written. This cell is the final executable ETL stage.

In [15]:
pipeline.validate(transformed)
pipeline.load(transformed)

output_manifest = []
for output_path in sorted(pipeline.PROCESSED_DIR.glob("*.csv")):
    output_manifest.append(
        {
            "Output file": output_path.name,
            "Rows": len(pd.read_csv(output_path)),
            "Size (KB)": round(output_path.stat().st_size / 1024, 1),
        }
    )

display(pd.DataFrame(output_manifest))
print("VALIDATION PASSED")
print(
    f"Public output: {len(transformed['latest'])} rows "
    "(3 indicators × 9 regions)"
)
print(
    "Synthetic contribution output: "
    f"{len(synthetic['region_summary'])} regional rows"
)

,Output file,Rows,Size (KB)
0,fusion21_map_metrics_synthetic.csv,63,26.400
1,fusion21_projects_synthetic.csv,45,9.600
2,fusion21_region_summary_synthetic.csv,9,1.200
3,imd_lad2019.csv,317,256.100
4,imd_rgn2019.csv,9,7.800
5,metrics_latest.csv,27,22.700
6,metrics_timeseries.csv,27,21.400
7,social_need_composite_latest.csv,9,9.000
8,unemployment_rgn_latest.csv,9,6.000


VALIDATION PASSED
Public output: 27 rows (3 indicators × 9 regions)
Synthetic contribution output: 9 regional rows


## 10. Run the website / 启动网站

After the notebook completes, run the website from a terminal in this folder:

```powershell
.venv\Scripts\python.exe pipeline.py --app-only --port 8501
```

On Windows, `启动网站.bat` performs the same action. The website reads only the
validated files written to `data/processed/`.

## 11. Interpretation and limitations / 解释与限制

- The workflow is reproducible and all nine English Regions are retained.
- Population weighting is used for IMD at both geographic aggregation stages.
- The social-need index is descriptive; it is not an official statistic or a
  causal impact estimate.
- IMD 2019 and the latest unemployment period do not share the same date.
- Foundation Score currently uses relative Min-Max scaling across nine regions.
- All Fusion21 data in this prototype are synthetic. Real business conclusions
  require approved, non-sensitive company data or direct staff feedback.